# 🤖 LLM 기반 자연어 쿼리 분해

사용자의 자연어 질문을 LLM이 **구조화된 검색 쿼리**로 변환하는 실습입니다.

## 📋 학습 목표

1. **자연어 → 구조화 쿼리**: LLM으로 검색어(search_text)와 필터(filter)를 자동 분리
2. **쿼리 분해**: 복잡한 질문을 여러 서브 쿼리로 쪼개서 검색
3. **키워드 검색의 한계 체감**: 왜 Agentic Retrieval이 필요한지 동기 부여

## 💡 핵심 아이디어

```
사용자: "10만원 이하 유아동 카테고리에서 선물하기 좋은 제품 추천해줘"

❌ 기존 방식: search_text에 전체 문장 그대로 전달 → 엉뚱한 결과
✅ LLM 분해: search_text="선물" + filter="category eq '유아동' and normal_price le 100000"
```

---

## 1️⃣ 라이브러리 임포트 및 초기화

In [ ]:
from azure.search.documents import SearchClient
from azure.core.credentials import AzureKeyCredential
from openai import AzureOpenAI
from dotenv import load_dotenv
import os
import json

# 환경 변수 로드
load_dotenv(override=True)

# Azure AI Search 클라이언트
search_endpoint = os.getenv("SEARCH_ENDPOINT")
search_key = os.getenv("SEARCH_ADMIN_KEY")
index_name = os.getenv("SEARCH_INDEX_NAME")

search_client = SearchClient(search_endpoint, index_name, AzureKeyCredential(search_key))

# Azure OpenAI 클라이언트
aoai_client = AzureOpenAI(
    azure_endpoint=os.getenv("AZURE_OPEN_AI_ENDPOINT"),
    api_key=os.getenv("AZURE_OPEN_AI_KEY"),
    api_version=os.getenv("AZURE_OPENAI_CHAT_API_VERSION")
)
chat_model = os.getenv("AZURE_OPENAI_CHAT_DEPLOYMENT")

print("✅ 클라이언트 초기화 완료")
print(f"   Search Index: {index_name}")
print(f"   Chat Model: {chat_model}")

---

## 2️⃣ 문제 체감: 자연어를 그대로 검색하면?

사용자의 자연어 질문을 `search_text`에 **그대로** 넣으면 어떤 결과가 나오는지 확인합니다.

In [ ]:
# 자연어 질문을 그대로 검색
natural_query = "10만원 이하 유아동 카테고리에서 선물하기 좋은 제품 추천해줘"

print(f"{'='*80}")
print(f"❌ 자연어 그대로 검색: '{natural_query}'")
print(f"{'='*80}\n")

results = search_client.search(
    search_text=natural_query,
    top=5,
    select=["title", "brand", "category", "normal_price"],
    include_total_count=True
)

print(f"📊 총 {results.get_count()}개 결과\n")

for idx, result in enumerate(results, 1):
    score = result['@search.score']
    print(f"{idx}. [점수: {score:.2f}] {result['title']}")
    print(f"   카테고리: {result['category']} | 가격: {result['normal_price']:,}원")
    print()

print("⚠️  '10만원', '이하', '추천해줘' 같은 단어가 토큰화되어")
print("   의도와 무관한 문서가 매칭됩니다.")
print("   카테고리 필터도 적용되지 않고, 가격 조건도 무시됩니다.")

---

## 3️⃣ LLM으로 자연어 → 구조화된 쿼리 변환

LLM에게 사용자의 자연어를 분석하여 `search_text`와 `filter`로 분리하도록 합니다.

### 인덱스 스키마 정보를 LLM에 전달

LLM이 올바른 필터를 생성하려면 **인덱스의 필드 정보**를 알아야 합니다:

| 필드 | 타입 | 필터 가능 | 사용 가능한 값 |
|------|------|-----------|---------------|
| `category` | String | ✅ | 유아동, 패션잡화, 이미용, 스포츠/레져, 주방, 문화/취미, 패션의류, 보석/장신구, 인테리어, 생활/건강, 언더웨어 |
| `brand` | String | ✅ | (다양한 브랜드명) |
| `normal_price` | Int32 | ✅ | 숫자 (원 단위) |

In [ ]:
QUERY_DECOMPOSE_SYSTEM_PROMPT = """당신은 이커머스 검색 시스템의 쿼리 분석기입니다.
사용자의 자연어 질문을 Azure AI Search 쿼리 파라미터로 변환하세요.

## 인덱스 스키마

- id: String (key)
- title: String (searchable, 한국어 분석기)
- brand: String (searchable, filterable, facetable)
- category: String (filterable, facetable)
  - 가능한 값: 유아동, 패션잡화, 이미용, 스포츠/레져, 주방, 문화/취미, 패션의류, 보석/장신구, 인테리어, 생활/건강, 언더웨어
- normal_price: Int32 (filterable, sortable)
- review: String (searchable, 한국어 분석기)

## 변환 규칙

1. **search_text**: 실제 검색할 키워드만 추출 (필터 조건 제외)
2. **filter**: OData 필터 문법으로 변환
   - 카테고리: category eq '값'
   - 가격 상한: normal_price le 값
   - 가격 하한: normal_price ge 값
   - 브랜드: brand eq '값'
   - 복합 조건: and/or로 연결
3. **order_by**: 정렬이 필요한 경우 (예: "저렴한 순" → "normal_price asc")

## 응답 형식 (반드시 JSON만 출력)

```json
{
  "search_text": "검색 키워드",
  "filter": "OData 필터 표현식 또는 null",
  "order_by": "정렬 표현식 또는 null",
  "reasoning": "변환 이유를 간단히 설명"
}
```
"""

def decompose_query(user_query: str) -> dict:
    """자연어 질문을 구조화된 검색 쿼리로 변환"""
    response = aoai_client.chat.completions.create(
        model=chat_model,
        messages=[
            {"role": "system", "content": QUERY_DECOMPOSE_SYSTEM_PROMPT},
            {"role": "user", "content": user_query}
        ],
        temperature=0,
        response_format={"type": "json_object"}
    )
    return json.loads(response.choices[0].message.content)

print("✅ 쿼리 분해 함수 정의 완료")

---

## 4️⃣ 쿼리 분해 테스트

다양한 자연어 질문으로 LLM의 분해 결과를 확인합니다.

In [ ]:
# 테스트 질문들
test_queries = [
    "10만원 이하 유아동 카테고리에서 선물하기 좋은 제품 추천해줘",
    "스포츠용 가방 중에서 가격 저렴한 순으로 보여줘",
    "이미용 제품 중 5만원 이상 20만원 이하",
    "노스페이스 자켓",
    "아기 출산 선물 세트 3만원대",
]

for query in test_queries:
    print(f"{'='*80}")
    print(f"💬 사용자 질문: \"{query}\"")
    print(f"{'='*80}")
    
    result = decompose_query(query)
    
    print(f"  🔍 search_text: {result.get('search_text')}")
    print(f"  🎯 filter:      {result.get('filter')}")
    print(f"  📊 order_by:    {result.get('order_by')}")
    print(f"  💡 reasoning:   {result.get('reasoning')}")
    print()

---

## 5️⃣ 비교: 자연어 그대로 vs LLM 분해 후 검색

동일한 질문에 대해 **두 가지 방식**의 검색 결과를 비교합니다.

In [ ]:
def search_and_display(search_text, filter_expr=None, order_by=None, top=5, label=""):
    """검색 실행 및 결과 출력 헬퍼"""
    kwargs = {
        "search_text": search_text,
        "top": top,
        "select": ["title", "brand", "category", "normal_price"],
        "include_total_count": True
    }
    if filter_expr:
        kwargs["filter"] = filter_expr
    if order_by:
        kwargs["order_by"] = [order_by]
    
    results = search_client.search(**kwargs)
    count = results.get_count()
    
    print(f"\n{label}")
    print(f"  search_text=\"{search_text}\"")
    if filter_expr:
        print(f"  filter=\"{filter_expr}\"")
    if order_by:
        print(f"  order_by=\"{order_by}\"")
    print(f"  → {count}개 결과\n")
    
    for idx, r in enumerate(results, 1):
        print(f"  {idx}. {r['title']}")
        print(f"     [{r['category']}] {r['brand']} | {r['normal_price']:,}원")
    
    if count == 0:
        print("  (결과 없음)")
    print()


# 비교 질문
user_question = "10만원 이하 유아동 카테고리에서 선물하기 좋은 제품 추천해줘"

print(f"{'='*80}")
print(f"💬 사용자 질문: \"{user_question}\"")
print(f"{'='*80}")

# 방법 1: 자연어 그대로
search_and_display(
    search_text=user_question,
    label="❌ 방법 1: 자연어 그대로 검색"
)

# 방법 2: LLM 분해 후
decomposed = decompose_query(user_question)
search_and_display(
    search_text=decomposed["search_text"],
    filter_expr=decomposed.get("filter"),
    order_by=decomposed.get("order_by"),
    label="✅ 방법 2: LLM 분해 후 검색"
)

In [ ]:
# 두 번째 비교: 정렬 조건이 포함된 질문
user_question = "스포츠 카테고리에서 가방 저렴한 순으로 보여줘"

print(f"{'='*80}")
print(f"💬 사용자 질문: \"{user_question}\"")
print(f"{'='*80}")

# 방법 1: 자연어 그대로
search_and_display(
    search_text=user_question,
    label="❌ 방법 1: 자연어 그대로 검색"
)

# 방법 2: LLM 분해 후
decomposed = decompose_query(user_question)
search_and_display(
    search_text=decomposed["search_text"],
    filter_expr=decomposed.get("filter"),
    order_by=decomposed.get("order_by"),
    label="✅ 방법 2: LLM 분해 후 검색"
)

---

## 6️⃣ 복잡한 질문 분해: 멀티 쿼리

하나의 질문 안에 **여러 검색 의도**가 포함된 경우, LLM이 여러 서브 쿼리로 분해합니다.

예: "유아동 선물 세트랑 스포츠 가방 둘 다 찾아줘" → 2개의 쿼리로 분해

In [ ]:
MULTI_QUERY_SYSTEM_PROMPT = """당신은 이커머스 검색 시스템의 쿼리 분석기입니다.
사용자의 복잡한 질문을 여러 개의 독립적인 검색 쿼리로 분해하세요.

## 인덱스 스키마

- title: String (searchable, 한국어 분석기)
- brand: String (searchable, filterable, facetable)
- category: String (filterable, facetable)
  - 가능한 값: 유아동, 패션잡화, 이미용, 스포츠/레져, 주방, 문화/취미, 패션의류, 보석/장신구, 인테리어, 생활/건강, 언더웨어
- normal_price: Int32 (filterable, sortable)
- review: String (searchable, 한국어 분석기)

## 규칙

1. 하나의 의도만 있으면 queries 배열에 1개만 반환
2. 여러 의도가 있으면 각각 독립적인 쿼리로 분해
3. 각 서브 쿼리에 의도(intent)를 명시

## 응답 형식 (반드시 JSON만 출력)

```json
{
  "original_query": "원래 질문",
  "queries": [
    {
      "intent": "이 서브 쿼리의 의도",
      "search_text": "검색 키워드",
      "filter": "OData 필터 또는 null",
      "order_by": "정렬 또는 null"
    }
  ]
}
```
"""

def decompose_multi_query(user_query: str) -> dict:
    """복잡한 질문을 여러 서브 쿼리로 분해"""
    response = aoai_client.chat.completions.create(
        model=chat_model,
        messages=[
            {"role": "system", "content": MULTI_QUERY_SYSTEM_PROMPT},
            {"role": "user", "content": user_query}
        ],
        temperature=0,
        response_format={"type": "json_object"}
    )
    return json.loads(response.choices[0].message.content)

print("✅ 멀티 쿼리 분해 함수 정의 완료")

In [ ]:
# 복잡한 질문 테스트
complex_queries = [
    "유아동 선물 세트랑 여성 가방 둘 다 찾아줘",
    "5만원 이하 스킨케어 제품이랑 10만원 이상 향수도 보여줘",
    "노스페이스 자켓 가격 높은 순으로, 그리고 아디다스 운동화도 찾아줘",
]

for query in complex_queries:
    print(f"\n{'='*80}")
    print(f"💬 사용자 질문: \"{query}\"")
    print(f"{'='*80}")
    
    result = decompose_multi_query(query)
    
    for i, sub in enumerate(result["queries"], 1):
        print(f"\n  📌 서브 쿼리 {i}: {sub['intent']}")
        print(f"     search_text: {sub['search_text']}")
        print(f"     filter:      {sub.get('filter')}")
        print(f"     order_by:    {sub.get('order_by')}")
    print()

In [ ]:
# 멀티 쿼리 실제 실행
user_question = "유아동 선물 세트랑 여성 가방 둘 다 찾아줘"

print(f"{'='*80}")
print(f"💬 사용자 질문: \"{user_question}\"")
print(f"{'='*80}")

decomposed = decompose_multi_query(user_question)

all_results = []
for i, sub in enumerate(decomposed["queries"], 1):
    print(f"\n{'─'*40}")
    print(f"📌 서브 쿼리 {i}: {sub['intent']}")
    print(f"{'─'*40}")
    
    search_and_display(
        search_text=sub["search_text"],
        filter_expr=sub.get("filter"),
        order_by=sub.get("order_by"),
        top=3,
        label=f"🔍 검색 실행"
    )

---

## 7️⃣ 직접 실습: 자유 질문

원하는 자연어 질문을 입력하여 LLM 분해 → 검색 결과를 확인해 보세요.

In [ ]:
# 👇 원하는 질문을 입력하세요
my_question = "배송 빠르고 품질 좋은 주방용품 5만원 이하로 찾아줘"

print(f"{'='*80}")
print(f"💬 내 질문: \"{my_question}\"")
print(f"{'='*80}")

# Step 1: LLM 분해
decomposed = decompose_query(my_question)
print(f"\n🤖 LLM 분해 결과:")
print(f"  search_text: {decomposed['search_text']}")
print(f"  filter:      {decomposed.get('filter')}")
print(f"  order_by:    {decomposed.get('order_by')}")
print(f"  reasoning:   {decomposed.get('reasoning')}")

# Step 2: 검색 실행
search_and_display(
    search_text=decomposed["search_text"],
    filter_expr=decomposed.get("filter"),
    order_by=decomposed.get("order_by"),
    top=10,
    label="\n🔍 검색 결과"
)

---

## 💡 정리: 이 방식의 한계와 Agentic Retrieval

### 우리가 구현한 것

```
사용자 자연어 → [LLM 분해] → search_text + filter + order_by → [Azure AI Search] → 결과
```

### 이 방식의 한계

| 한계 | 설명 |
|------|------|
| **프롬프트 관리** | 스키마가 바뀌면 프롬프트도 수동 수정 필요 |
| **필터 오류** | LLM이 잘못된 OData 문법을 생성할 수 있음 |
| **단일 인덱스** | 여러 인덱스를 조합한 검색이 어려움 |
| **대화 맥락** | 멀티턴 대화에서 \"그것보다 저렴한 것\" 같은 참조 처리 어려움 |
| **검색 방식 고정** | 키워드/벡터/하이브리드 중 동적 선택 불가 |

### Agentic Retrieval이 해결하는 것

Azure AI Search의 **Agentic Retrieval**은 위 과정을 **플랫폼 레벨**에서 자동화합니다:

- ✅ 인덱스 스키마를 자동으로 인식 (프롬프트에 하드코딩 불필요)
- ✅ 유효한 OData 필터를 보장
- ✅ 여러 인덱스를 Knowledge Base로 통합
- ✅ 대화 히스토리를 자동 반영
- ✅ 쿼리 특성에 따라 검색 방식을 동적 선택
- ✅ Reasoning Effort로 품질/비용 조절

👉 자세한 내용은 [Agentic Retrieval 실습 레포](https://github.com/ChangJu-Ahn/ignite25-LAB511-build-agentic-knowledge-bases-next-level-rag-with-azure-ai-search)를 참고하세요.

---

## 🧭 다음 단계

| ⬅️ 이전 | 🏠 목차 | ➡️ 다음 |
|:---------|:-------:|--------:|
| [Lab 07: Vision 데이터 보강](../07-enriched_dataset/01-enrich_with_vision.ipynb) | [워크숍 홈](../README.md) | [Lab 10: 이미지 기반 검색](../10-image_based_search/01-image_based_search.ipynb) |